# SREG — Exploración paso a paso

Esta notebook te permite ver exactamente qué hace cada parte del sistema.

**Secciones:**
1. Crear un mundo (la red bayesiana)
2. Visualizar la estructura causal
3. Samplear la "verdad oculta"
4. Jugar un episodio paso a paso
5. Puntuar al agente
6. Comparar teacher vs agente random

In [ ]:
# Setup — importar todo lo necesario
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

from sreg.tools.world_gen import WorldGenTool, WorldGenConfig
from sreg.tools.world_check import WorldCheckTool
from sreg.solver.exact_bayes import ExactBayesSolver
from sreg.tools.episode_gen import EpisodeGenTool, EpisodeGenConfig
from sreg.tools.task_gen import TaskGenTool
from sreg.models.task import TaskSpec, TaskType
from sreg.env.episode import EpisodeRunner
from sreg.models.episode import Action, ActionType
from sreg.tools.verifier import VerifierTool

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print("✅ Todo importado correctamente")

## 1. Crear un mundo

Un "mundo" es una red bayesiana: nodos conectados por flechas causales, cada uno con probabilidades condicionales.

**Parámetros que podés tocar:**
- `seed` — cambia esto para generar mundos distintos
- `num_nodes` — cuántos nodos en total (3-20)
- `edge_strength` — qué tan fuerte es la relación causa-efecto (0.1=ruidoso/difícil, 1.0=determinístico/fácil)
- `num_states` — cuántos estados tiene cada nodo (2=binario, 3=low/medium/high, etc.)

In [ ]:
# ===== CAMBIÁ ESTOS PARÁMETROS PARA EXPLORAR =====
SEED = 5
NUM_NODES = 6
EDGE_STRENGTH = 0.7   # 0.1 = difícil, 0.7 = fácil, 1.0 = casi determinístico
NUM_STATES = 3         # 2 = binario, 3 = low/med/high
EPISODE_SEED = 0
BUDGET = 4
# ==================================================

gen = WorldGenTool()
config = WorldGenConfig(
    seed=SEED,
    num_nodes=NUM_NODES,
    edge_strength=EDGE_STRENGTH,
    num_states=NUM_STATES,
)
world = gen.generate(config)

print(f"Mundo: {world.id}")
print(f"Dificultad: {world.difficulty.level}")
print(f"\nNodos ({len(world.nodes)}):")
for n in world.nodes:
    emoji = {"latent": "🔒", "observable": "👁️", "target": "🎯"}[n.type]
    print(f"  {emoji} {n.name} ({n.type}) — estados: {n.states}")

print(f"\nConexiones causales ({len(world.edges)}):")
for e in world.edges:
    print(f"  {e.from_node}  ──→  {e.to_node}")

## 2. Visualizar la estructura causal

Dibujamos el grafo. Los colores indican el tipo de nodo:
- 🔴 Rojo = **latente** (oculto, el agente no puede ver esto)
- 🟢 Verde = **observable** (el agente puede elegir mirarlo, gasta presupuesto)
- 🟡 Amarillo = **target** (lo que hay que predecir)

In [ ]:
# Construir grafo de networkx
dag = nx.DiGraph()
for node in world.nodes:
    dag.add_node(node.name, type=node.type)
for edge in world.edges:
    dag.add_edge(edge.from_node, edge.to_node)

# Colores por tipo
color_map = {"latent": "#FF6B6B", "observable": "#51CF66", "target": "#FFD43B"}
node_colors = [color_map[dag.nodes[n]["type"]] for n in dag.nodes]

# Layout jerárquico (causa arriba, efectos abajo)
pos = nx.shell_layout(dag)

fig, ax = plt.subplots(figsize=(10, 7))
nx.draw(
    dag, pos, ax=ax,
    with_labels=True,
    node_color=node_colors,
    node_size=2500,
    font_size=9,
    font_weight="bold",
    edge_color="#555",
    arrows=True,
    arrowsize=20,
    connectionstyle="arc3,rad=0.1",
)

# Leyenda
patches = [
    mpatches.Patch(color="#FF6B6B", label="Latente (oculto)"),
    mpatches.Patch(color="#51CF66", label="Observable (el agente puede mirar)"),
    mpatches.Patch(color="#FFD43B", label="Target (hay que predecir)"),
]
ax.legend(handles=patches, loc="upper left", fontsize=10)
ax.set_title(f"Estructura causal — {world.id}", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Validar el mundo

El `WorldCheckTool` revisa que el mundo sea interesante y válido:
- ¿Es un DAG? (no tiene ciclos)
- ¿Tiene nodos ocultos?
- ¿Los observables están conectados al target?
- ¿La entropía del target es suficiente? (que no sea trivial de adivinar)
- ¿Hay d-separaciones? (independencias condicionales interesantes)

In [ ]:
checker = WorldCheckTool()
result = checker.check(world)

if result.passed:
    print("✅ El mundo pasó todas las validaciones")
else:
    print("❌ El mundo tiene problemas:")
    for f in result.failures:
        print(f"  - {f}")

print("\nMétricas:")
for k, v in result.metrics.items():
    print(f"  {k}: {v:.3f}")

## 4. Samplear la "verdad oculta"

Ahora "tiramos los dados". El sistema recorre los nodos en orden causal y samplea un valor para cada uno según las probabilidades condicionales.

Esto genera **una realidad concreta**: cada nodo tiene un valor. El agente no sabe estos valores — tiene que descubrirlos observando.

In [ ]:
solver = ExactBayesSolver(world)
true_state = solver.sample_state(seed=EPISODE_SEED)

print("La verdad oculta (el agente NO ve esto):\n")
for node in world.nodes:
    emoji = {"latent": "🔒", "observable": "👁️", "target": "🎯"}[node.type]
    print(f"  {emoji} {node.name} = {true_state[node.name]}")

print(f"\n🎯 El agente tiene que predecir: target_outcome = {true_state['target_outcome']}")

## 5. Ver la distribución ANTES de observar nada

Esto es lo que el agente "cree" antes de mirar nada — la distribución prior del target.

Si la prior ya da la respuesta correcta con alta probabilidad, el mundo es demasiado fácil. Si es casi uniforme, hay bastante incertidumbre para reducir.

In [ ]:
prior = solver.posterior("target_outcome")
prior_entropy = solver.entropy(prior)

fig, ax = plt.subplots(figsize=(8, 4))
states = list(prior.keys())
probs = list(prior.values())
bars = ax.bar(states, probs, color=["#74C0FC", "#74C0FC", "#74C0FC"], edgecolor="black")

# Marcar la verdad
true_idx = states.index(true_state["target_outcome"])
bars[true_idx].set_color("#FF6B6B")
bars[true_idx].set_label(f"Verdad: {true_state['target_outcome']}")

for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{p:.1%}", ha="center", fontsize=11)

ax.set_ylim(0, 1.1)
ax.set_ylabel("Probabilidad")
ax.set_title(f"Prior P(target_outcome) — antes de observar nada\nEntropía: {prior_entropy:.2f} bits")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

best_guess = max(prior, key=prior.get)
print(f"Sin observar nada, el agente diría: '{best_guess}' ({prior[best_guess]:.1%})")
print(f"La verdad es: '{true_state['target_outcome']}'")
print(f"{'✅ Adivinaía bien!' if best_guess == true_state['target_outcome'] else '❌ Se equivocaría — necesita observar.'}")

## 6. El teacher juega el episodio paso a paso

Ahora el teacher (el jugador perfecto) va a observar variables una por una. En cada turno:
1. Calcula cuánta **información ganaría** observando cada variable disponible
2. Elige la que más información da
3. La observa y actualiza su creencia sobre el target

Vas a ver cómo la distribución cambia turno a turno.

In [ ]:
# Crear episodio y runner
ep_tool = EpisodeGenTool()
episode = ep_tool.generate(world, EpisodeGenConfig(budget=BUDGET, seed=EPISODE_SEED))
runner = EpisodeRunner(world, episode, true_state)

obs_nodes = list(episode.available_nodes)
evidence = {}
history = []  # guardar cada paso para graficar después

# Guardar el prior como paso 0
history.append({
    "step": 0,
    "label": "Prior (sin observar)",
    "posterior": dict(prior),
    "entropy": prior_entropy,
    "info_gain": 0,
    "observed": None,
    "observed_value": None,
})

print(f"Budget: {BUDGET} observaciones")
print(f"Variables disponibles: {obs_nodes}")
print(f"Verdad: target_outcome = {true_state['target_outcome']}")
print("=" * 60)

for step_num in range(BUDGET):
    available = [n for n in obs_nodes if n not in evidence]
    if not available:
        break

    # Mostrar info gain de cada opción
    print(f"\n--- Turno {step_num + 1} ---")
    print("Info gain esperado por variable:")
    gains = {}
    for node in available:
        g = solver.information_gain("target_outcome", evidence, node)
        gains[node] = g
        print(f"  {node}: {g:.4f} bits")

    # El teacher elige la mejor
    output = solver.optimal_action("target_outcome", evidence, available)
    if output.recommended_action is None:
        print("  → Entropía ya es ~0, no vale la pena observar más.")
        break

    node = output.recommended_action.node
    result = runner.step(output.recommended_action)
    evidence[result.observation.node] = result.observation.state

    post = solver.posterior("target_outcome", evidence)
    h = solver.entropy(post)
    map_state = max(post, key=post.get)
    correct = map_state == true_state["target_outcome"]

    print(f"\n  👉 Elige: {node}")
    print(f"  👀 Observa: {node} = {result.observation.state}")
    print(f"  📊 P(target) = { {k: f'{v:.1%}' for k, v in post.items()} }")
    print(f"  🧠 Mejor predicción: {map_state} {'✅' if correct else '❌'}")
    print(f"  📉 Entropía: {h:.3f} bits")

    history.append({
        "step": step_num + 1,
        "label": f"Observa {node}={result.observation.state}",
        "posterior": dict(post),
        "entropy": h,
        "info_gain": output.information_gain,
        "observed": node,
        "observed_value": result.observation.state,
    })

# Resultado final
final = history[-1]
final_map = max(final["posterior"], key=final["posterior"].get)
print("\n" + "=" * 60)
print(f"Predicción final del teacher: {final_map}")
print(f"Verdad: {true_state['target_outcome']}")
print(f"{'✅ ¡Correcto!' if final_map == true_state['target_outcome'] else '❌ Incorrecto (pasa ~10% de las veces)'}")

## 7. Visualizar cómo cambia la creencia turno a turno

Dos gráficos:
- **Arriba**: Cómo cambia la distribución P(target) en cada paso
- **Abajo**: Cómo baja la entropía (incertidumbre) a medida que observa

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), gridspec_kw={"height_ratios": [3, 1.5]})

# --- Gráfico 1: Evolución de la distribución ---
states = list(history[0]["posterior"].keys())
n_steps = len(history)
x = np.arange(n_steps)
width = 0.25

colors_states = {"low": "#74C0FC", "medium": "#B197FC", "high": "#FF8787"}

for i, state in enumerate(states):
    vals = [h["posterior"][state] for h in history]
    bars = ax1.bar(x + i * width, vals, width, label=state, color=colors_states.get(state, "#aaa"), edgecolor="black", linewidth=0.5)

# Línea horizontal en la probabilidad de la verdad
ax1.axhline(y=0.5, color="gray", linestyle="--", alpha=0.3)

labels = [h["label"] for h in history]
ax1.set_xticks(x + width)
ax1.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax1.set_ylabel("Probabilidad")
ax1.set_ylim(0, 1.05)
ax1.set_title(f"Evolución de P(target_outcome) — verdad: {true_state['target_outcome']}", fontsize=13)
ax1.legend(title="Estado", fontsize=10)

# --- Gráfico 2: Entropía ---
entropies = [h["entropy"] for h in history]
ax2.plot(range(n_steps), entropies, "o-", color="#E03131", linewidth=2, markersize=8)
ax2.fill_between(range(n_steps), entropies, alpha=0.15, color="#E03131")
ax2.set_xticks(range(n_steps))
ax2.set_xticklabels([f"Paso {h['step']}" for h in history], fontsize=9)
ax2.set_ylabel("Entropía (bits)")
ax2.set_title("Incertidumbre sobre el target a lo largo del episodio", fontsize=12)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

## 8. Puntuar al agente

Supongamos que un agente (como un LLM) juega el mismo episodio pero observa las variables **en orden aleatorio** en vez de elegir la más informativa. ¿Cómo se compara con el teacher?

In [ ]:
# --- Teacher: ya lo tenemos ---
teacher_final = history[-1]["posterior"]
teacher_evidence = dict(evidence)

# --- Agente random: observa en orden aleatorio ---
rng = np.random.default_rng(42)
random_order = rng.permutation(obs_nodes).tolist()

episode2 = ep_tool.generate(world, EpisodeGenConfig(budget=BUDGET, seed=1))
runner2 = EpisodeRunner(world, episode2, true_state)
random_evidence = {}
random_history = [{"step": 0, "posterior": dict(prior), "entropy": prior_entropy}]

for i, node in enumerate(random_order[:BUDGET]):
    result2 = runner2.step(Action(type=ActionType.OBSERVE, node=node))
    random_evidence[node] = result2.observation.state
    post2 = solver.posterior("target_outcome", random_evidence)
    h2 = solver.entropy(post2)
    random_history.append({"step": i + 1, "posterior": dict(post2), "entropy": h2})

random_final = random_history[-1]["posterior"]

# --- Comparar con VerifierTool ---
verifier = VerifierTool()
true_post = solver.posterior("target_outcome", {**teacher_evidence})

teacher_score = verifier.score(
    agent_posterior=teacher_final,
    true_posterior=true_post,
    budget_used=len(teacher_evidence),
    budget_total=BUDGET,
)

random_true_post = solver.posterior("target_outcome", random_evidence)
random_score = verifier.score(
    agent_posterior=random_final,
    true_posterior=random_true_post,
    budget_used=len(random_evidence),
    budget_total=BUDGET,
)

print("Comparación Teacher vs Random:\n")
print(f"{'':20} {'Teacher':>12} {'Random':>12}")
print(f"{'-'*46}")

teacher_map = max(teacher_final, key=teacher_final.get)
random_map = max(random_final, key=random_final.get)
truth = true_state["target_outcome"]

print(f"{'Predicción':20} {teacher_map:>12} {random_map:>12}")
print(f"{'Verdad':20} {truth:>12} {truth:>12}")
print(f"{'Correcto?':20} {'✅' if teacher_map == truth else '❌':>12} {'✅' if random_map == truth else '❌':>12}")
print(f"{'KL divergence':20} {teacher_score.functional_score:>12.4f} {random_score.functional_score:>12.4f}")
print(f"{'Entropía final':20} {history[-1]['entropy']:>12.3f} {random_history[-1]['entropy']:>12.3f}")
print(f"\n(KL divergence más bajo = mejor — 0 es perfecto)")

In [ ]:
# Gráfico: entropía teacher vs random
fig, ax = plt.subplots(figsize=(10, 5))

teacher_ent = [h["entropy"] for h in history]
random_ent = [h["entropy"] for h in random_history]

ax.plot(range(len(teacher_ent)), teacher_ent, "o-", color="#2B8A3E", linewidth=2, markersize=8, label="Teacher (óptimo)")
ax.plot(range(len(random_ent)), random_ent, "s--", color="#E03131", linewidth=2, markersize=8, label="Random")
ax.fill_between(range(len(teacher_ent)), teacher_ent, alpha=0.1, color="#2B8A3E")
ax.fill_between(range(len(random_ent)), random_ent, alpha=0.1, color="#E03131")

ax.set_xlabel("Paso")
ax.set_ylabel("Entropía (bits) — menor = más seguro")
ax.set_title("Teacher vs Random: reducción de incertidumbre por paso")
ax.legend(fontsize=11)
ax.set_xticks(range(max(len(teacher_ent), len(random_ent))))
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 9. Accuracy del teacher en muchos episodios

¿Qué pasa si corremos muchos mundos y muchos episodios? El teacher debería acertar >90% de las veces.

(Esto tarda ~5 segundos)

In [ ]:
from sreg.models.world import NodeType

n_worlds = 30
n_episodes = 5
correct = 0
total = 0
per_world_acc = []

for w_seed in range(n_worlds):
    w = gen.generate(WorldGenConfig(seed=w_seed, num_nodes=6, edge_strength=EDGE_STRENGTH))
    s = ExactBayesSolver(w)
    obs = [n.name for n in w.nodes if n.type == NodeType.OBSERVABLE]
    w_correct = 0

    for ep_seed in range(n_episodes):
        ts = s.sample_state(seed=w_seed * 1000 + ep_seed)
        _, traj = s.generate_trajectory("target_outcome", obs, len(obs), seed=w_seed * 1000 + ep_seed)
        final_post = traj[-1].posterior
        prediction = max(final_post, key=final_post.get)
        if prediction == ts["target_outcome"]:
            correct += 1
            w_correct += 1
        total += 1

    per_world_acc.append(w_correct / n_episodes)

accuracy = correct / total
print(f"Accuracy del teacher: {accuracy:.1%} ({correct}/{total})")
print(f"{'✅ Supera 90%' if accuracy > 0.90 else '⚠️ Por debajo de 90%'}")

# Gráfico
fig, ax = plt.subplots(figsize=(12, 4))
colors = ["#2B8A3E" if a >= 0.8 else "#E8590C" for a in per_world_acc]
ax.bar(range(n_worlds), [a * 100 for a in per_world_acc], color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(y=90, color="red", linestyle="--", label="Umbral 90%")
ax.set_xlabel("Mundo (seed)")
ax.set_ylabel("Accuracy (%)")
ax.set_title(f"Accuracy del teacher por mundo ({n_episodes} episodios cada uno) — Total: {accuracy:.1%}")
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## 10. Efecto de edge_strength en la dificultad

¿Qué pasa cuando bajamos `edge_strength`? Las relaciones causales se vuelven más ruidosas, y el teacher acierta menos.

In [ ]:
strengths = [0.2, 0.4, 0.6, 0.8, 1.0]
accs_by_strength = []

for es in strengths:
    correct = 0
    total = 0
    for w_seed in range(20):
        w = gen.generate(WorldGenConfig(seed=w_seed, num_nodes=6, edge_strength=es))
        s = ExactBayesSolver(w)
        obs = [n.name for n in w.nodes if n.type == NodeType.OBSERVABLE]
        for ep_seed in range(5):
            ts = s.sample_state(seed=w_seed * 1000 + ep_seed)
            _, traj = s.generate_trajectory("target_outcome", obs, len(obs), seed=w_seed * 1000 + ep_seed)
            prediction = max(traj[-1].posterior, key=traj[-1].posterior.get)
            if prediction == ts["target_outcome"]:
                correct += 1
            total += 1
    accs_by_strength.append(correct / total)
    print(f"edge_strength={es:.1f}  →  accuracy={correct/total:.1%}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(strengths, [a * 100 for a in accs_by_strength], "o-", color="#364FC7", linewidth=2, markersize=10)
ax.axhline(y=90, color="red", linestyle="--", alpha=0.5, label="Umbral 90%")
ax.axhline(y=33.3, color="gray", linestyle=":", alpha=0.5, label="Azar (3 estados)")
ax.fill_between(strengths, [a * 100 for a in accs_by_strength], alpha=0.1, color="#364FC7")
ax.set_xlabel("Edge strength")
ax.set_ylabel("Accuracy del teacher (%)")
ax.set_title("Dificultad controlable: edge_strength bajo = más difícil")
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## 11. Preview del dataset (lo que exportaríamos)

Así se ve una entrada del dataset que generaría Phase 7. Cada fila es un paso de un episodio jugado por el teacher.

In [ ]:
# Armar un ejemplo de dataset entry
dataset_entry = {
    "world_id": world.id,
    "world_seed": SEED,
    "episode_seed": EPISODE_SEED,
    "template": world.template_family,
    "difficulty": world.difficulty.level,
    "target_node": "target_outcome",
    "true_target_value": true_state["target_outcome"],
    "num_nodes": len(world.nodes),
    "num_observable": sum(1 for n in world.nodes if n.type == NodeType.OBSERVABLE),
    "trajectory": [],
}

for h in history:
    step_entry = {
        "step": h["step"],
        "observed_node": h["observed"],
        "observed_value": h["observed_value"],
        "posterior": {k: round(v, 4) for k, v in h["posterior"].items()},
        "entropy": round(h["entropy"], 4),
        "map_prediction": max(h["posterior"], key=h["posterior"].get),
    }
    dataset_entry["trajectory"].append(step_entry)

print("Ejemplo de entrada del dataset (JSONL):\n")
print(json.dumps(dataset_entry, indent=2))

---

## Para explorar

Volvé arriba y cambiá los parámetros:

- `SEED = 42` → otro mundo completamente distinto
- `EDGE_STRENGTH = 0.3` → mundo más difícil (relaciones ruidosas)
- `EDGE_STRENGTH = 0.95` → mundo fácil (relaciones casi determinísticas)
- `NUM_NODES = 8` → más variables, más complejo
- `NUM_STATES = 2` → nodos binarios (sí/no)
- `EPISODE_SEED = 5` → misma estructura, distintos "dados"